In [4]:
import requests as r
import json
import csv
import pandas as pd
from pandas import json_normalize

#url to post
PokeDexURL = 'https://pokeapi.co/api/v2/pokedex/1/'
#get data from that URL
res = r.get(PokeDexURL)
#PokeDex (National)

#put the data coming in from res into values as json
values = res.json()

#Define variables
PokeDex = [] 
MaxDexNumFound = 0
MaxDexNum = 0
CurrentNum = 0
PokeDexID = ''
PokemonName = ''

#Define headers
PokeDex.append(['PokeDexID|PokemonName'])

#Load table
while MaxDexNumFound == 0:
    #Check to see if PokeDexID exists 
    try:
        #If PokeDexID exists, add to extract
        PokeDexID =str(values['pokemon_entries'][CurrentNum]['entry_number'])
        PokemonName = values['pokemon_entries'][CurrentNum]['pokemon_species']['name'].title()
        PokeDex.append([PokeDexID + '|' + PokemonName])
        CurrentNum += 1
        #If PokeDexID does not exist, print the max PokeDexID found
    except:
        print('Max PokeDexID Found:' + str(CurrentNum))
        MaxDexNumFound = 1
        MaxDexNum = CurrentNum

Max PokeDexID Found:1025


In [8]:
PokeDex[0:10]

[['PokeDexID|PokemonName'],
 ['1|Bulbasaur'],
 ['2|Ivysaur'],
 ['3|Venusaur'],
 ['4|Charmander'],
 ['5|Charmeleon'],
 ['6|Charizard'],
 ['7|Squirtle'],
 ['8|Wartortle'],
 ['9|Blastoise']]

In [ ]:
#https://www.tcgplayer.com/search/pokemon/product?Rarity=Illustration+Rare|Special+Illustration+Rare&Price_Condition=Less+Than&advancedSearch=true&productLineName=pokemon&view=grid&page=1
#https://www.tcgplayer.com/search/pokemon/product?productLineName=pokemon&q=trainer+gallery&view=grid&page=1
#https://www.tcgplayer.com/search/pokemon-japan/product?productLineName=pokemon-japan&view=grid&RarityName=Art+Rare|Special+Art+Rare&page=1
#https://www.tcgplayer.com/search/pokemon-japan/product?productLineName=pokemon-japan&view=grid&RarityName=Character+Rare|Character+Super+Rare&page=1

In [ ]:
Extract = ['ProductPK','Series','CardName','CardNumber','Rarity','SpotlightPrice']
ProductPK = [['452021'],['509983'],['618701'],['6187015555555']]
BaseURL = 'https://www.tcgplayer.com/product/'
FailedPKChecks = 0

from selenium import webdriver
from selenium.webdriver.common.by import By
import time

# Set up the WebDriver
driver = webdriver.Chrome()

for PK in ProductPK:

    # Open the news website
    driver.get(BaseURL + PK[0])

    tempList = []

    # Allow the page to load
    time.sleep(5)

    print('ProductPK: ' + PK[0])
    tempList.append(PK[0])

    ############---------BreadCrumbList

    # Try searching the next PK
    try: 
        BreadCrumbList = driver.find_elements(By.CLASS_NAME, "tcg-breadcrumbs__list")[0].text
        ProductType = BreadCrumbList.split('\n')[1]
    except:
        'PK Site Not Found'
        ProductType = 'InvalidPK'
        FailedPKChecks = FailedPKChecks + 1

    #Check to see if product is a Pokemon card
    if ProductType == 'Pokemon Cards':

        # Split the BreadCrumbList
        tempList.append(BreadCrumbList.split('\n')[2])                                      #Series
        tempList.append(BreadCrumbList.split('\n')[3])                                      #CardName

        ############---------ItemDetails

        # Search for the ItemDetails
        ItemDetails = driver.find_elements(By.CLASS_NAME, "product__item-details__content")[0].text

        # Split the ItemDetails
        tempList.append(ItemDetails.split('\n')[1].split(' / ')[1].split(':')[1])           #CardNumber
        tempList.append(ItemDetails.split('\n')[1].split(' / ')[2])                         #Rarity

        ############---------SpotlightPrice

        # Search for the SpotlightPrice
        SpotlightPrice = driver.find_elements(By.CLASS_NAME, "spotlight__price")[0].text    #Removes the dollar sign
        tempList.append(SpotlightPrice[1:len(SpotlightPrice)])

        Extract.append(tempList)

# Close the browser
driver.quit()

ProductPK: 452021
ProductPK: 509983
ProductPK: 618701
ProductPK: 6187015555555


In [103]:
Extract

['ProductPK',
 'Series',
 'CardName',
 'CardNumber',
 'Rarity',
 'SpotlightPrice',
 ['452021',
  'SWSH12: Silver Tempest Trainer Gallery',
  'Rockruff',
  'TG07/TG30',
  'Ultra Rare',
  '2.97'],
 ['509983',
  'SV03: Obsidian Flames',
  'Pidgeot ex - 225/197',
  '225/197',
  'Special Illustration Rare',
  '13.52']]

# Look For New Cards

In [20]:
with open("MaxPK.txt", "r") as f:
    MaxPK = int(f.readlines()[0])
#MaxPK = 42346
LastSuccessfulPK = MaxPK
Extract = [] 
Headers = ['ProductPK','Series','CardName','CardNumber','Rarity','SpotlightPrice','InsertedDTM','LastModifiedDTM']
Extract.append(Headers)
BaseURL = 'https://www.tcgplayer.com/product/'

In [41]:
FailedPKChecks = 0
SearchMarker = 0
SearchLimit = 5000

from selenium import webdriver
from selenium.webdriver.common.by import By
import time

CurrentDTM = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

# Set up the WebDriver
driver = webdriver.Chrome()

while (SearchMarker < SearchLimit and FailedPKChecks < 3000):
    
    print('[' + time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()) + '] ' + str(SearchMarker + 1) + ' out of ' + str(SearchLimit) +  ' - ProductPK: ' + str(MaxPK))

    # Open the website
    driver.get(BaseURL + str(MaxPK))

    tempList = []

    # Allow the page to load
    time.sleep(5)

    tempList.append(str(MaxPK))

    ############---------BreadCrumbList

    # Try searching the next PK
    try: 
        BreadCrumbList = driver.find_elements(By.CLASS_NAME, "tcg-breadcrumbs__list")[0].text
        ProductType = BreadCrumbList.split('\n')[1]
        LastSuccessfulPK = MaxPK
        FailedPKChecks = 0
    except:
        'PK Site Not Found'
        ProductType = 'InvalidPK'
        FailedPKChecks = FailedPKChecks + 1

    #Check to see if product is a Pokemon card
    if (ProductType == 'Pokemon Cards' or ProductType == 'Pokemon Japan'):

        # Split the BreadCrumbList
        tempList.append(BreadCrumbList.split('\n')[2])                                      #Series
        tempList.append(BreadCrumbList.split('\n')[3])                                      #CardName

        ############---------ItemDetails

        # Search for the ItemDetails
        ItemDetails = driver.find_elements(By.CLASS_NAME, "product__item-details__content")[0].text

        # Split the ItemDetails
        tempList.append(ItemDetails.split('\n')[1].split(' / ')[1].split(':')[1])           #CardNumber
        tempList.append(ItemDetails.split('\n')[1].split(' / ')[2])                         #Rarity

        ############---------SpotlightPrice

        # Search for the SpotlightPrice
        SpotlightPrice = driver.find_elements(By.CLASS_NAME, "spotlight__price")[0].text    #Removes the dollar sign
        tempList.append(SpotlightPrice[1:len(SpotlightPrice)])

        tempList.append(CurrentDTM)                                                         #InsertedDTM
        tempList.append(CurrentDTM)                                                         #LastModifiedDTM

        Extract.append(tempList)

    
    MaxPK = MaxPK + 1
    SearchMarker = SearchMarker + 1

# Close the browser
driver.quit()

with open("MaxPK.txt", "w") as f:
    f.write(str(LastSuccessfulPK))

[2025-06-01 02:00:12] 1 out of 5000 - ProductPK: 45474
[2025-06-01 02:00:18] 2 out of 5000 - ProductPK: 45475
[2025-06-01 02:00:23] 3 out of 5000 - ProductPK: 45476
[2025-06-01 02:00:28] 4 out of 5000 - ProductPK: 45477
[2025-06-01 02:00:33] 5 out of 5000 - ProductPK: 45478
[2025-06-01 02:00:38] 6 out of 5000 - ProductPK: 45479
[2025-06-01 02:00:44] 7 out of 5000 - ProductPK: 45480
[2025-06-01 02:00:49] 8 out of 5000 - ProductPK: 45481
[2025-06-01 02:00:54] 9 out of 5000 - ProductPK: 45482
[2025-06-01 02:00:59] 10 out of 5000 - ProductPK: 45483
[2025-06-01 02:01:04] 11 out of 5000 - ProductPK: 45484
[2025-06-01 02:01:09] 12 out of 5000 - ProductPK: 45485
[2025-06-01 02:01:15] 13 out of 5000 - ProductPK: 45486
[2025-06-01 02:01:20] 14 out of 5000 - ProductPK: 45487
[2025-06-01 02:01:25] 15 out of 5000 - ProductPK: 45488
[2025-06-01 02:01:30] 16 out of 5000 - ProductPK: 45489
[2025-06-01 02:01:35] 17 out of 5000 - ProductPK: 45490
[2025-06-01 02:01:40] 18 out of 5000 - ProductPK: 45491
[

KeyboardInterrupt: 

In [25]:
LastSuccessfulPK

45167

In [42]:
len(Extract)

1

In [27]:
Extract

[['ProductPK',
  'Series',
  'CardName',
  'CardNumber',
  'Rarity',
  'SpotlightPrice',
  'InsertedDTM',
  'LastModifiedDTM'],
 ['42568',
  'Base Set 2',
  'Water Energy',
  '130/130',
  'Common',
  '0.01',
  '2025-05-31 15:55:47',
  '2025-05-31 15:55:47'],
 ['43053',
  'Deck Exclusives',
  'Rayquaza - 22/107',
  '022/107',
  'Holo Rare',
  '1.74',
  '2025-05-31 15:55:47',
  '2025-05-31 15:55:47'],
 ['44418',
  'Fossil',
  'Aerodactyl (1)',
  '01/62',
  'Holo Rare',
  '6.00',
  '2025-05-31 19:11:38',
  '2025-05-31 19:11:38'],
 ['44419',
  'Fossil',
  'Lapras (10)',
  '10/62',
  'Holo Rare',
  '10.24',
  '2025-05-31 19:11:38',
  '2025-05-31 19:11:38'],
 ['44420',
  'Fossil',
  'Magneton (11)',
  '11/62',
  'Holo Rare',
  '3.64',
  '2025-05-31 19:11:38',
  '2025-05-31 19:11:38'],
 ['44421',
  'Fossil',
  'Moltres (12)',
  '12/62',
  'Holo Rare',
  '149.99',
  '2025-05-31 19:11:38',
  '2025-05-31 19:11:38'],
 ['44422',
  'Fossil',
  'Muk (13)',
  '13/62',
  'Holo Rare',
  '4.64',
  '2025

In [37]:
with open("MaxPK.txt", "w") as f:
    f.write(str(MaxPK))

In [15]:
#First is 42346
MaxPK

42627

In [ ]:
with open("Extract.txt", "w") as f:
    for row in Extract:
        f.write(','.join(str(item) for item in row ) + '\n')

# Update Old Cards

In [1]:
import csv

with open("Extract.txt", newline='\n') as f:
    reader = csv.reader(f)
    Data = list(reader)

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

BaseURL = 'https://www.tcgplayer.com/product/'

CurrentDTM = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

# Set up the WebDriver
driver = webdriver.Chrome()

for i in range(1,len(Data)):
    print('[' + time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()) + '] ' + str(i) + ' out of ' + str(len(Data)) +  ' - ProductPK: ' + Data[i][0])
    print(Data[i])
    
    # Open the website
    driver.get(BaseURL + str(Data[i][0]))

    # Allow the page to load
    time.sleep(5)

    BreadCrumbList = driver.find_elements(By.CLASS_NAME, "tcg-breadcrumbs__list")[0].text

    # Search for the SpotlightPrice
    SpotlightPrice = driver.find_elements(By.CLASS_NAME, "spotlight__price")[0].text    #Removes the dollar sign
    Data[i][5] = SpotlightPrice
    
    # Update LastModifiedDTM
    Data[i][7] = CurrentDTM

# Close the browser
driver.quit()

[2025-06-05 18:39:50] 1 out of 319 - ProductPK: 42346
['42346', 'Base Set', 'Alakazam', '001/102', 'Holo Rare', '$12.20', '2025-05-31 01:22:10', '2025-05-31 15:03:08']
[2025-06-05 18:39:56] 2 out of 319 - ProductPK: 42347
['42347', 'Base Set', 'Mewtwo', '010/102', 'Holo Rare', '$12.99', '2025-05-31 01:22:10', '2025-05-31 15:03:08']
[2025-06-05 18:40:01] 3 out of 319 - ProductPK: 42348
['42348', 'Base Set', 'Lightning Energy', '100/102', 'Common', '$0.05', '2025-05-31 01:22:10', '2025-05-31 15:03:08']
[2025-06-05 18:40:06] 4 out of 319 - ProductPK: 42349
['42349', 'Base Set', 'Psychic Energy', '101/102', 'Common', '$0.49', '2025-05-31 01:22:10', '2025-05-31 15:03:08']
[2025-06-05 18:40:11] 5 out of 319 - ProductPK: 42350
['42350', 'Base Set', 'Water Energy', '102/102', 'Common', '$0.34', '2025-05-31 01:22:10', '2025-05-31 15:03:08']
[2025-06-05 18:40:16] 6 out of 319 - ProductPK: 42351
['42351', 'Base Set', 'Nidoking', '011/102', 'Holo Rare', '$24.40', '2025-05-31 01:22:10', '2025-05-31

InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=137.0.7151.56)
Stacktrace:
	GetHandleVerifier [0x0x7ff6e46ffea5+79173]
	GetHandleVerifier [0x0x7ff6e46fff00+79264]
	(No symbol) [0x0x7ff6e44b9e5a]
	(No symbol) [0x0x7ff6e44a5c25]
	(No symbol) [0x0x7ff6e44cac44]
	(No symbol) [0x0x7ff6e45403c5]
	(No symbol) [0x0x7ff6e4560922]
	(No symbol) [0x0x7ff6e4538743]
	(No symbol) [0x0x7ff6e45014c1]
	(No symbol) [0x0x7ff6e4502253]
	GetHandleVerifier [0x0x7ff6e49ca2dd+3004797]
	GetHandleVerifier [0x0x7ff6e49c472d+2981325]
	GetHandleVerifier [0x0x7ff6e49e3380+3107360]
	GetHandleVerifier [0x0x7ff6e471aa2e+188622]
	GetHandleVerifier [0x0x7ff6e47222bf+219487]
	GetHandleVerifier [0x0x7ff6e4708df4+115860]
	GetHandleVerifier [0x0x7ff6e4708fa9+116297]
	GetHandleVerifier [0x0x7ff6e46ef558+11256]
	BaseThreadInitThunk [0x0x7ffcee96e8d7+23]
	RtlUserThreadStart [0x0x7ffcf061c5dc+44]


In [5]:
Data[-1]

['45167',
 'Jungle',
 'Poke Ball',
 '64/64',
 'Common',
 '0.14',
 '2025-05-31 19:11:38',
 '2025-05-31 19:11:38']

# Append Extract To Data

In [29]:
Extract[0]

['ProductPK',
 'Series',
 'CardName',
 'CardNumber',
 'Rarity',
 'SpotlightPrice',
 'InsertedDTM',
 'LastModifiedDTM']

In [31]:
data[0]

['ProductPK',
 'Series',
 'CardName',
 'CardNumber',
 'Rarity',
 'SpotlightPrice',
 'InsertedDTM',
 'LastModifiedDTM']

In [32]:
for i in range(2, len(Extract)):
    data.append(Extract[i])

In [35]:
len(data)

319

# Search > Sources > Page > TCGPlayer-cdn.tcgplayer.com

In [8]:
import requests as r
import json
import time
import datetime
import matplotlib.pyplot as plt
import pandas as pd
from pandas.io.json import json_normalize

#url to post
action_postURL = 'https://infinite-api.tcgplayer.com/price/history/271837?range=annual'

#get data from that URL
res = r.get(action_postURL)

#put the data coming in from res into values as json
values = res.json()

ImportError: cannot import name 'json_normalize' from 'pandas.io.json' (c:\Users\Romeo\anaconda3\Lib\site-packages\pandas\io\json\__init__.py)

In [4]:
r.get(action_postURL)

<Response [200]>